#Assignment#4
**Name**: Muhammad Usman Tahir\
**Roll No**: 221463\
**Class**: Deep Learning

#Dataset Selected: UCF50

In [1]:
import kagglehub

dataset_path = kagglehub.dataset_download("pypiahmad/realistic-action-recognition-ucf50")
print("Dataset downloaded to:", dataset_path)

100%|██████████| 3.04G/3.04G [02:25<00:00, 22.4MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/pypiahmad/realistic-action-recognition-ucf50/versions/1


In [2]:
import os

os.listdir(dataset_path)

['UCF50']

In [3]:
DATASET_PATH = os.path.join(dataset_path, "UCF50")
os.listdir(DATASET_PATH)


['Nunchucks',
 'Skijet',
 'HorseRiding',
 'TaiChi',
 'Biking',
 'SkateBoarding',
 'HighJump',
 'Skiing',
 'JumpingJack',
 'HorseRace',
 'HulaHoop',
 'BenchPress',
 'PlayingViolin',
 'Rowing',
 'MilitaryParade',
 'BreastStroke',
 'YoYo',
 'Lunges',
 'Diving',
 'WalkingWithDog',
 'Basketball',
 'SoccerJuggling',
 'TrampolineJumping',
 'PlayingGuitar',
 'Billiards',
 'PommelHorse',
 'PizzaTossing',
 'Swing',
 'Punch',
 'Mixing',
 'Fencing',
 'BaseballPitch',
 'JumpRope',
 'PullUps',
 'CleanAndJerk',
 'TennisSwing',
 'JavelinThrow',
 'PlayingPiano',
 'GolfSwing',
 'PushUps',
 'RopeClimbing',
 'JugglingBalls',
 'RockClimbingIndoor',
 'Kayaking',
 'Drumming',
 'SalsaSpin',
 'PlayingTabla',
 'VolleyballSpiking',
 'ThrowDiscus',
 'PoleVault']

##Choosing only 5 actions

In [4]:
SELECTED_ACTIONS = [
    "WalkingWithDog",
    "JumpingJack",
    "PushUps",
    "Punch",
    "Biking"
]

for a in SELECTED_ACTIONS:
    print(a, "=", os.path.exists(os.path.join(DATASET_PATH, a)))

WalkingWithDog = True
JumpingJack = True
PushUps = True
Punch = True
Biking = True


In [5]:
import shutil

for folder in os.listdir(DATASET_PATH):
    if folder not in SELECTED_ACTIONS:
        shutil.rmtree(os.path.join(DATASET_PATH, folder))

In [6]:
os.listdir(DATASET_PATH)

['Biking', 'JumpingJack', 'WalkingWithDog', 'Punch', 'PushUps']

##Extracting Frames

In [7]:
import shutil
shutil.rmtree("processed_data", ignore_errors=True)

In [8]:
import os
import cv2

PROCESSED_DIR = "processed_data"
FRAMES = 10
IMG_SIZE = 160

os.makedirs(PROCESSED_DIR, exist_ok=True)

for action in os.listdir(DATASET_PATH):
    action_path = os.path.join(DATASET_PATH, action)
    save_action_path = os.path.join(PROCESSED_DIR, action)
    os.makedirs(save_action_path, exist_ok=True)

    videos = os.listdir(action_path)[:60]  # limit per class

    for video in videos:
        cap = cv2.VideoCapture(os.path.join(action_path, video))
        frames = []

        while len(frames) < FRAMES:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
            frames.append(frame)

        cap.release()

        if len(frames) == FRAMES:
            video_name = video.split(".")[0]
            video_dir = os.path.join(save_action_path, video_name)
            os.makedirs(video_dir, exist_ok=True)

            for i, frame in enumerate(frames):
                cv2.imwrite(
                    os.path.join(video_dir, f"frame_{i}.jpg"),
                    frame
                )

In [9]:
os.listdir(PROCESSED_DIR)

['Biking', 'JumpingJack', 'WalkingWithDog', 'Punch', 'PushUps']

In [10]:
sample = os.listdir(os.path.join(PROCESSED_DIR, "PushUps"))[0]
os.listdir(os.path.join(PROCESSED_DIR, "PushUps", sample))

['frame_0.jpg',
 'frame_9.jpg',
 'frame_3.jpg',
 'frame_4.jpg',
 'frame_8.jpg',
 'frame_2.jpg',
 'frame_7.jpg',
 'frame_1.jpg',
 'frame_5.jpg',
 'frame_6.jpg']

##Build CNN + LSTM Model

In [12]:
import os
import cv2
import numpy as np
import tensorflow as tf

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import TimeDistributed, LSTM, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Sequential
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

In [13]:
PROCESSED_DIR = "processed_data"
IMG_SIZE = 160
FRAMES = 10

In [14]:
ACTIONS = sorted(os.listdir(PROCESSED_DIR))

le = LabelEncoder()
le.fit(ACTIONS)

NUM_CLASSES = len(ACTIONS)

print(ACTIONS)
print("Classes:", NUM_CLASSES)

['Biking', 'JumpingJack', 'Punch', 'PushUps', 'WalkingWithDog']
Classes: 5


In [15]:
cnn = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

cnn.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [16]:
model = Sequential([
    TimeDistributed(cnn, input_shape=(FRAMES, IMG_SIZE, IMG_SIZE, 3)),
    TimeDistributed(GlobalAveragePooling2D()),
    LSTM(64),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation="softmax")
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [19]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed                │ (None, 10, 5, 5, 1280) │     2,257,984 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 10, 1280)       │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │       344,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,602,629 (9.93 MB)

 Trainable params: 344,645 (1.31 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

##Data Generator

In [20]:
def data_generator(data_dir, actions, label_encoder):
    while True:
        for action in actions:
            action_path = os.path.join(data_dir, action)
            for video in os.listdir(action_path):
                frames = []
                video_path = os.path.join(action_path, video)

                for i in range(FRAMES):
                    frame_path = os.path.join(video_path, f"frame_{i}.jpg")
                    frame = cv2.imread(frame_path)
                    frame = frame / 255.0
                    frames.append(frame)

                X = np.array([frames])
                y = label_encoder.transform([action])
                y = to_categorical(y, NUM_CLASSES)

                yield X, y

##Model Traning

In [21]:
def count_samples(data_dir):
    return sum(
        len(os.listdir(os.path.join(data_dir, action)))
        for action in os.listdir(data_dir)
    )

steps_per_epoch = count_samples(PROCESSED_DIR)
print("Steps:", steps_per_epoch)

Steps: 300


In [23]:
history = model.fit(
    data_generator(PROCESSED_DIR, ACTIONS, le),
    steps_per_epoch=steps_per_epoch,
    epochs=15
)

Epoch 1/15
300/300 ━━━━━━━━━━━━━━━━━━━━ 17s 57ms/step - accuracy: 0.6603 - loss: 1.0079
Epoch 2/15
300/300 ━━━━━━━━━━━━━━━━━━━━ 16s 52ms/step - accuracy: 0.6511 - loss: 1.1009
Epoch 3/15
300/300 ━━━━━━━━━━━━━━━━━━━━ 16s 55ms/step - accuracy: 0.7422 - loss: 0.8185
Epoch 4/15
300/300 ━━━━━━━━━━━━━━━━━━━━ 16s 52ms/step - accuracy: 0.7742 - loss: 0.7663
Epoch 5/15
300/300 ━━━━━━━━━━━━━━━━━━━━ 16s 54ms/step - accuracy: 0.7723 - loss: 0.6909
Epoch 6/15
300/300 ━━━━━━━━━━━━━━━━━━━━ 16s 54ms/step - accuracy: 0.7570 - loss: 0.6951
Epoch 7/15
300/300 ━━━━━━━━━━━━━━━━━━━━ 16s 52ms/step - accuracy: 0.8421 - loss: 0.4992
Epoch 8/15
300/300 ━━━━━━━━━━━━━━━━━━━━ 16s 52ms/step - accuracy: 0.8523 - loss: 0.4702
Epoch 9/15
300/300 ━━━━━━━━━━━━━━━━━━━━ 15s 51ms/step - accuracy: 0.8975 - loss: 0.3734
Epoch 10/15
300/300 ━━━━━━━━━━━━━━━━━━━━ 16s 54ms/step - accuracy: 0.8757 - loss: 0.3883
Epoch 11/15
300/300 ━━━━━━━━━━━━━━━━━━━━ 15s 52ms/step - accuracy: 0.9107 - loss: 0.2992
Epoch 12/15
300/300 ━━━━━━━━━━

In [24]:
model.save("action_model.h5")

import pickle
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)